In [0]:
%pip install geopandas shapely pyproj fiona
dbutils.library.restartPython()

In [0]:
import geopandas as gpd
from pyspark.sql.functions import col, current_timestamp

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
dbutils.fs.cp(
    "abfss://data@scottishhousingdl.dfs.core.windows.net/raw/school_catchments/pub_schlsn.shp",
    "file:///tmp/pub_schlsn.shp"
)
dbutils.fs.cp(
    "abfss://data@scottishhousingdl.dfs.core.windows.net/raw/school_catchments/pub_schlsn.dbf",
    "file:///tmp/pub_schlsn.dbf"
)
dbutils.fs.cp(
    "abfss://data@scottishhousingdl.dfs.core.windows.net/raw/school_catchments/pub_schlsn.shx",
    "file:///tmp/pub_schlsn.shx"
)
dbutils.fs.cp(
    "abfss://data@scottishhousingdl.dfs.core.windows.net/raw/school_catchments/pub_schlsn.prj",
    "file:///tmp/pub_schlsn.prj"
)

gdf = gpd.read_file("/tmp/pub_schlsn.shp")

print(gdf.shape)
print(gdf.dtypes)
gdf.head(5)

In [0]:
gdf["geometry_wkt"] = gdf["geometry"].apply(lambda g: g.wkt if g else None)
gdf["crs"] = "EPSG:27700"
gdf = gdf.drop(columns=["geometry"])

print(gdf.columns.tolist())
gdf.head(3)

In [0]:
import re

def clean_col_name(name):
    name = name.lower()
    name = re.sub(r"[^\w]", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

gdf.columns = [clean_col_name(c) for c in gdf.columns]
print(gdf.columns.tolist())

In [0]:
import pandas as pd
df_bronze = spark.createDataFrame(gdf)
df_bronze.printSchema()
df_bronze.select("la_s_code", "local_auth", "seed_code", "school_nam", "crs").show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.school_catchments")
)

In [0]:
# Source: Spatial Hub Scotland - School Catchments (Secondary Non-Denominational)
# File: pub_schlsn.shp, uploaded to ADLS at raw/school_catchments/
# CRS: EPSG:27700 (British National Grid)
# Geometry stored as WKT string in geometry_wkt column.
# 328 secondary non-denominational school catchment polygons, all Scottish councils.
# Licence: UK Open Government Licence (OGL)